# 07 — Un sistema multiagente que busca predictores para NAS

**Caso de estudio aparte**, no uno de los bloques del taller. Antes de ver los
agentes hay que entender el problema, porque es lo que explica por qué un agente
tiene sentido aquí.

Se corre de arriba abajo y **tarda entre 12 y 15 minutos**, casi todos en las
celdas del Experimentador y del Orquestador. Necesita Ollama arriba y los
paquetes de `requirements.txt`: `numpy`, `scikit-learn` y `scipy`.

---

## ¿Qué es NAS?

Diseñar una red neuronal es, todavía hoy, trabajo manual de alguien con
experiencia: cuántas capas, de qué tipo, con qué tamaño de kernel, cuántos
canales. Se prueba, se mide, se ajusta, se vuelve a probar.

**Neural Architecture Search** (NAS) automatiza esa búsqueda. En lugar de que
una persona proponga arquitecturas, un algoritmo las genera, las evalúa y se
queda con las mejores.

Tres piezas lo definen:

| | |
|---|---|
| **Espacio de búsqueda** | qué arquitecturas son posibles — el catálogo de piezas y cómo se combinan |
| **Estrategia de búsqueda** | cómo se recorre ese espacio: aleatoria, evolutiva, por gradiente, por refuerzo |
| **Estrategia de evaluación** | cómo se decide qué tan buena es una arquitectura candidata |

Las dos primeras son relativamente baratas. **La tercera es el problema.**

## El cuello de botella: evaluar cuesta entrenar

Para saber si una arquitectura es buena hay que **entrenarla**. Y entrenar una
red es caro: horas de GPU por candidato.

Un espacio de búsqueda realista tiene millones o miles de millones de
arquitecturas válidas. Si cada evaluación cuesta horas, la búsqueda es
imposible — no por falta de ideas, sino de cómputo.

Toda la investigación en NAS de los últimos años gira alrededor de esta pregunta:
**¿cómo estimar qué tan buena es una arquitectura sin pagar el costo de
entrenarla?**

## El caso concreto: super-resolución

Este caso de estudio viene de un problema específico: **super-resolución de
imágenes** (SR). Es la tarea de tomar una imagen de baja resolución y
reconstruir una versión de alta resolución — recuperar detalle que no está en
los píxeles originales.

Es un buen banco de pruebas para NAS porque es exigente en las dos direcciones
a la vez: la red tiene que reconstruir detalle fino y ser lo bastante
liviana para correr en un dispositivo real.

### Cómo se mide

**PSNR** (*Peak Signal-to-Noise Ratio*), en decibeles. Compara la imagen
reconstruida con la original y mide qué tan parecidas son. Más alto es mejor.
En este dataset la mediana es **32.4 dB** y tres cuartas partes caen entre 28 y
35; hay una cola de 27 arquitecturas por debajo de 20 dB, que son redes que
sencillamente no aprendieron.

Pero PSNR solo no alcanza, porque la red más pesada casi siempre reconstruye
mejor. Por eso el problema es **multiobjetivo** — tres objetivos a la vez:

| objetivo | dirección | qué mide |
|---|---|---|
| **PSNR** | maximizar | calidad de la reconstrucción |
| **Parámetros** | minimizar | cuánta memoria ocupa |
| **FLOPs** | minimizar | cuánto cómputo necesita para inferir |

No hay una única "mejor" arquitectura: hay un **frente de Pareto**, un conjunto
de arquitecturas donde no se puede mejorar un objetivo sin empeorar otro. La
búsqueda las recorre con un algoritmo evolutivo, NSGA-III.

### Cómo se representa una arquitectura

El espacio de búsqueda se llama **BASS**, y cada arquitectura se codifica como
un **cromosoma de 28 tokens**, cada uno con un valor de 0 a 7:

```
[1, 6, 5, 0, 0, 4, 4, 1, 2, 0, 3, 0, 7, 7, 0, 2, 7, 7, 7, 3, 6, 2, 5, 4, 5, 1, 3, 3]
```

| token | qué define |
|---|---|
| el primero | el número de canales global de la red |
| los 27 restantes | nueve grupos de operación (tres por rama), cada uno con **tipo de operación** (convolución estándar, dilatada, *depthwise*, *inverted bottleneck*…), **tamaño de kernel** (1×1, 3×3, 5×5, 7×7) y **cuántas veces se repite** |

Los valores son **códigos Gray de 3 bits**: están elegidos para que arquitecturas
vecinas difieran en un solo bit, lo que hace que las mutaciones del algoritmo
evolutivo produzcan cambios pequeños y no saltos arbitrarios.

Ese esquema cubre **más de 9×10¹⁸ arquitecturas válidas**. Con horas de GPU por
evaluación, recorrerlo entrenando cada candidato no es una opción.

### De dónde sale ese 9×10¹⁸

No sale de los bits. Si los 28 tokens fueran libres con 8 valores cada uno
serían `8²⁸ = 2⁸⁴ ≈ 1.9×10²⁵`, que es cuatro millones de veces más grande.

Sale de los **catálogos de piezas**, porque no todos los tokens tienen ocho
opciones reales:

| pool | opciones |
|---|---|
| Tipo de operación | **8** — conv 2D, dilatadas ×2/×3/×4, *depthwise separable*, *inverted bottleneck*, transpuesta, identidad |
| Tamaño de kernel | **4** — 1, 3, 5, 7 |
| Repeticiones | **4** — 1, 2, 3, 4 |
| Canales (token global) | **4** — 16, 32, 48, 64 |

Entonces cada **grupo de operación** tiene `8 × 4 × 4 = 128 = 2⁷`
configuraciones, y hay **nueve** grupos (tres ramas × tres operaciones):

```
128⁹ = 2⁶³ = 9 223 372 036 854 775 808 ≈ 9.22×10¹⁸
```

Ese es el número que cita el paper. Sumando el token de canales, `4 × 2⁶³ = 2⁶⁵
≈ 3.7×10¹⁹` — de ahí el "**more than** 9×10¹⁸".

### Y una consecuencia que importa para el predictor

Si el kernel solo tiene 4 opciones, ¿por qué se codifica con 3 bits, que dan 8?

Porque los códigos Gray usan el espacio completo y el mapeo a las opciones
reales es **dos a uno**. Esa redundancia es deliberada: garantiza que *cualquier*
mutación de un solo bit caiga en una arquitectura válida, que era el objetivo
del diseño.

El efecto secundario es que **el predictor ve distinciones que no existen**: dos
valores de token distintos pueden significar el mismo kernel de 5×5. Y le pasa a
**tres de los cuatro roles**: canales, kernel y repeticiones tienen cuatro
opciones reales cada uno, y solo el tipo de operación usa sus ocho.

Eso **no se ve en los datos**, y la celda siguiente lo deja claro: las 28
posiciones usan los ocho valores de token, así que desde el CSV la redundancia
es invisible. Se sabe por los catálogos de piezas, no por la columna.

## La salida: un predictor de rendimiento

La idea es simple de enunciar: **entrenar unas cuantas arquitecturas de verdad,
y usar esos resultados para entrenar un modelo que prediga el rendimiento de
las demás.**

Ese modelo —un regresor común, de los que se entrenan en segundos— se llama
**predictor de rendimiento** o *surrogate*. Recibe el cromosoma de 28 tokens y
devuelve un PSNR estimado. Con él, el algoritmo evolutivo puede evaluar 50 000
candidatos en minutos en lugar de en años.

Los números del caso, que son lo que vuelve viable todo esto:

| | |
|---|---|
| Generar el dataset: entrenar 1193 arquitecturas de verdad | **~9 500 horas de GPU** |
| Entrenar un predictor sobre ese dataset | **0.02 a 6 segundos de CPU** |
| Evaluar 50 000 candidatos durante la búsqueda, ya con el predictor | **~5.5 horas** |
| Hacerlo entrenando cada uno | inviable |

El costo caro **se paga una sola vez**. Después, el predictor se reutiliza para
búsquedas nuevas sin volver a entrenar nada.

Y de ahí sale la pregunta que este caso ataca:

> **¿Cuál regresor funciona mejor como predictor de rendimiento, y cómo lo
> sabemos con seguridad?**

Es una pregunta de experimentación pura: probar familias de modelos, medirlas
con el mismo protocolo, compararlas honestamente. Exactamente el tipo de trabajo
repetitivo, acotado y verificable donde un agente puede ser útil.

## Por qué esto es un buen caso para agentes

**El bucle del problema es el bucle agéntico.** Proponer un predictor,
entrenarlo, leer la métrica, decidir qué probar después. Es la misma forma de
los pasos 2 a 4, aplicada a algo donde el agente elige qué hacer y no solo
consulta.

**La evaluación es barata y objetiva.** Segundos por experimento, sin tocar la
GPU que ocupa el modelo de lenguaje. El agente puede correr veinte pruebas en el
tiempo que toma explicarlas.

**Y tenemos la respuesta publicada.** Eso es rarísimo: casi nunca se puede
*calificar* a un agente. Aquí sí.

### Los datos

De [Sarmiento-Rosales, Sosa Hernández y Monroy (2026)](https://doi.org/10.1016/j.swevo.2025.102236),
*"Evolutionary Neural Architecture Search for Super-Resolution: Benchmarking
SynFlow and model-based predictors"*, Swarm and Evolutionary Computation.
Dataset público bajo licencia MIT:
[MoNAS-SR-Predictors-Dataset](https://github.com/SergioSarmientoRosales/MoNAS-SR-Predictors-Dataset).

**1193 arquitecturas**, cada una entrenada 5 épocas sobre DIV2K y medida. El
paper compara once regresores y reporta sus resultados en su Tabla 3, que es
contra lo que vamos a comparar al agente.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")
from nas_agente import laboratorio as lab
from nas_agente import agentes
importlib.reload(lab); importlib.reload(agentes)

lab.limpiar_pizarron()
print("datos:", lab.DATOS.name)
print("X:", lab.X_CRUDO.shape, " y:", lab.Y.shape)

# Son arreglos de numpy, no DataFrames: para mirarlos se usa [:n], no .head(n).
lab.vistazo(4)

> **¿Editaste `src/nas_agente/agentes.py`?** Python no relee un módulo que ya
> cargó: tu cambio está en el disco y el kernel sigue con la versión vieja en
> memoria. Corre la celda siguiente y vuelve a llamar al agente.
>
> Usa **esa** celda y no la de setup: la de setup también recarga `laboratorio`,
> y eso **borra el pizarrón** —`PIZARRON = []` se vuelve a ejecutar—, así que
> perderías las evaluaciones y tendrías que correr el Experimentador otra vez.


In [ ]:
# Recargar SOLO los agentes, despues de editar agentes.py.
# No toca laboratorio, asi que el pizarron -lo ya evaluado- sobrevive.
import importlib
from nas_agente import agentes
importlib.reload(agentes)

# Para comprobar que si agarro tu cambio, mira el prompt que quedo cargado:
print(f"{len(lab.PIZARRON)} evaluaciones en el pizarron (intactas)")
print("SYSTEM del Experimentador, tal como esta AHORA en memoria:")
print(f"  {agentes.SYSTEM_EXPERIMENTADOR[:160]}...")


In [ ]:
# El tamano del espacio, y la redundancia de la codificacion.
import numpy as np

por_grupo = 8 * 4 * 4                      # operacion x kernel x repeticiones
print(f"  por grupo de operacion : {por_grupo} = 2^7")
print(f"  nueve grupos           : {por_grupo}^9 = 2^63 = {por_grupo**9:,}")
print(f"                         = {por_grupo**9:.3e}   <- el 9x10^18 del paper")
print(f"  + el token de canales  : 4 x 2^63 = 2^65 = {4*por_grupo**9:.3e}")
print(f"\n  si los 28 tokens fueran libres: 8^28 = {8**28:.3e}  (no es el caso)")

X = lab.X_CRUDO.astype(int)
roles = ["canales"] + ["tipo", "kernel", "repeticiones"] * 9
print(f"\n  valores distintos observados por posicion del cromosoma:")
for rol in ("canales", "tipo", "kernel", "repeticiones"):
    pos = [j for j, r in enumerate(roles) if r == rol]
    distintos = {len(np.unique(X[:, j])) for j in pos}
    print(f"    {rol:<13} ({len(pos):>2} posiciones): {distintos}")

print(f"\n  Las 28 posiciones usan los 8 valores: la redundancia NO se ve aqui.")
print(f"  Se sabe por los catalogos de piezas: canales, kernel y repeticiones")
print(f"  tienen 4 opciones reales con 8 valores de token -el mapeo es dos a")
print(f"  uno- y solo el tipo de operacion usa los 8. Es el precio de los")
print(f"  codigos Gray, y explica por que el one-hot de 224 columnas sale")
print(f"  peor mas abajo.")

## Los cuatro agentes

<img src="imagenes/18-los-cuatro.png" width="720">

**Los cuatro trabajan sobre un solo pizarrón**, y lo que los separa es qué
puede tocar cada uno. El trabajo se acumula ahí: el Analista lee los datos pero
no los cambia, el Experimentador es el único con las manos en la máquina, y el
Estadístico agrupa lo que los otros midieron sin volver a medir nada.

Fíjate en el que está de espaldas, en primer plano. Es el más alto y **no
alcanza el pizarrón**: tiene los brazos ocupados con los tres tubos que van a
los otros. No puede tocar nada por sí mismo, y eso no es un defecto del dibujo
— es el diseño.

| Agente | Responsabilidad | Sus tools |
|---|---|---|
| **Analista** | mira los datos y reporta lo que encuentra | solo lectura |
| **Experimentador** | entrena y evalúa predictores | el único que gasta CPU |
| **Estadístico** | dice si las diferencias son reales | solo lectura |
| **Orquestador** | reparte el trabajo y concluye | **los otros tres**, y `ranking` |

El Orquestador usa el patrón **agente-como-tool**: tres de sus cuatro tools no
son funciones de Python, son los otros agentes. El handoff queda explícito y
visible. La cuarta, `ranking`, solo ordena lo que los otros ya midieron, y
está ahí por un motivo que mide la sección del Experimentador.

El que de verdad gana su lugar es el **Estadístico**, y es poco común: su
trabajo es *desconfiar de los otros dos*. Sin él, el sistema reporta el máximo
de una lista y lo llama ganador.

## 1. El Analista

Pregúntale por los datos. Fíjate en las líneas `describir_dataset(...)`: lo que
reporta sale de ahí, no de su memoria.

Pero antes de correrlo, vale la pena mirar **con qué** lo estamos corriendo.
`ficha()` imprime las tres cosas que convierten al modelo en *este* agente: su
system prompt completo, la lista de tools que puede ejecutar y el formato de
salida que se le pide.

Esto importa más de lo que parece: **el modelo es el mismo para los cuatro
agentes**. No hay un modelo "analista" y otro "estadístico". Lo único que los
distingue es el contenido de esta ficha — texto que escribimos nosotros.

Cada uno de los cuatro agentes abre su sección con su ficha, y leerlas en
orden es medio notebook: lo que cambia de una a otra es *exactamente* lo que
hace que se comporten distinto. Y `ficha("analista", crudo=True)` agrega el
JSON exacto de las tools, que es lo que de verdad viaja en el request.

<img src="imagenes/14-analista.png" width="560">

**Solo tiene tools de lectura: ve y no toca.** Señala la hoja de datos desde
afuera de la vitrina, con la libreta en la otra mano para anotar lo que ve.

La huella marcada en el vidrio es de las veces que ya intentó tocarla. Es el
mismo vidrio de la imagen del bloque 1: lo que el modelo *pide* está de un lado,
lo que se *ejecuta* está del otro.


In [ ]:
# Que recibe el Analista para hacer su trabajo. Lee el system prompt y la
# lista de tools ANTES de ver la respuesta: casi todo lo que el agente haga
# despues esta escrito aqui, y lo que pase sin estar aqui es lo interesante.
agentes.ficha("analista")


In [ ]:
agentes.analista()

### Ver el bucle por dentro

Arriba viste el resumen: qué tools pidió y qué contestó. Con `detalle=True` se
ve la otra mitad — **qué se le manda al modelo en cada vuelta y qué devuelve**.

Esta celda contesta la pregunta que siempre sale en el bloque 2: *¿el modelo se
acuerda de lo anterior?* **No.** Fíjate en la línea `SE ENVIA` de cada vuelta:
en la 1 van 2 mensajes, en la 2 van 4. En cada vuelta se manda **la
conversación entera otra vez**. Lo que parece memoria es tu programa
volviéndole a contar todo, cada vez.

Y los tokens no son un estimado: salen del `prompt_eval_count` que devuelve
Ollama en cada llamada. Míralos subir, porque el contexto real de `qwen3.5:9b`
en el aula son **4096 tokens** — no los 262144 que anuncia la ficha del modelo.


In [ ]:
# detalle=True imprime cada vuelta completa; ver=False apaga el resumen, para
# no ver lo mismo dos veces. Son dos vueltas: una para pedir la tool y otra
# para redactar con lo que la tool devolvio.
agentes.analista("Cuantas arquitecturas hay y cuantas se repiten?",
                 detalle=True, ver=False)


### La línea del system prompt que hace que eso funcione

El Analista empieza con esta instrucción, y **no es adorno**:

> *"NUNCA respondas sin haber llamado una tool primero: no conoces este dataset
> de memoria y cualquier cifra que no venga de una tool es inventada."*

Medido: **sin esa línea, el Analista no llama ninguna tool 5 de 5 veces**, y
responde que el dataset tiene *"1000 imágenes de 256×256 píxeles con tokens en
el rango [0, 255] y PSNR promedio de 32.5 dB"*. Todo inventado, y siempre lo
mismo, porque la temperatura es 0.

Compáralo con el notebook 02, donde el system prompt decía *"no inventes
números"* y el modelo inventó igual. La diferencia no es la insistencia:

| | |
|---|---|
| *"no inventes"* | una **prohibición**. Falló. |
| *"siempre llama una tool primero, porque no conoces esto"* | una **instrucción con su razón**. 5/5. |

La otra hipótesis —que la tool no se llamara por no tener parámetros— se probó
y quedó descartada: con parámetro y sin él, el resultado fue idéntico. **Lo
decisivo fue el prompt.**

## 2. El Experimentador

El único que gasta CPU. Su system prompt insiste en algo que no es obvio:

> *"LA MÉTRICA QUE IMPORTA ES c_index, no mse."*

En NAS no interesa acertar el PSNR exacto: interesa **ordenar** arquitecturas,
porque de eso depende a cuál selecciona el algoritmo evolutivo. Un predictor que
se equivoca 2 dB en todo pero ordena bien, sirve.

Y hay una trampa en los datos: en la Tabla 3 del paper, **SVR gana por MSE y
Extra Trees gana por C-index**. Un agente que optimice MSE —lo que hace todo
tutorial de regresión— elige el modelo equivocado sin que nada se lo señale.

**Qué va a evaluar.** El catálogo tiene **18 predictores**: los **11 de la Tabla
3 del paper** —para poder contrastar contra un resultado publicado— y **7 más**
que el paper no probó, uno por familia que allí falta:

| agregado | qué aporta |
|---|---|
| `hist_gradient_boosting` | boosting por histogramas, el pariente de XGBoost |
| `gradient_boosting` | el boosting clásico de scikit-learn |
| `bagging` | bagging sobre árboles, sin la aleatoriedad extra de Random Forest |
| `bayesian_ridge` | lineal, con la regularización estimada de los datos |
| `huber` | lineal **robusto a extremos** — viene al caso: hay 27 PSNR bajo 20 dB |
| `mlp` | una red neuronal, que es la pregunta que siempre sale |
| `gaussian_process` | el surrogate clásico de optimización bayesiana |

Barrer los 18 cuesta **53 segundos de CPU**. Las 1193 arquitecturas sobre las
que se entrenan costaron **~9500 horas de GPU**. Ese desbalance es la razón de
ser de los predictores.

> `xgboost` es el único de los once que no viene con scikit-learn, y son 100 MB
> de descarga. Por eso **no está en `requirements.txt`**: si no lo tienes, el
> catálogo no lo ofrece y te dice cómo instalarlo. Se habilita con
> `pip install xgboost`.

<img src="imagenes/15-experimentador.png" width="560">

**El único con los brazos metidos en la máquina.** Es el único agente que gasta
CPU; los otros tres miran.

Esa tira de resultados que sale de la máquina y se le acumula en el piso es lo
que van a leer los demás. Él no decide quién gana: solo mide y entrega la tira.


In [ ]:
# La ficha del Experimentador, y la pregunta que vale la pena hacerse aqui:
# contra la del Analista, el modelo es el mismo, el bucle es el mismo y los
# guardrails son los mismos. Cambian dos cosas -el system prompt y la lista
# de tools- y con eso ya es otro agente.
#
# Fijate ademas en que las dos unicas tools de todo el sistema que gastan CPU
# -evaluar_predictor y buscar_hiperparametros- son suyas. Estan aqui y en
# ningun otro lado, a proposito.
agentes.ficha("experimentador")


In [ ]:
# Tarda unos dos minutos. La peticion no nombra ningun predictor a proposito:
# esa lista se la tiene que preguntar al catalogo, y eso es lo primero que hace.
#
# Mira cuantas tools pide por vuelta. Son 20 llamadas en CINCO vueltas, no en
# 20: el catalogo, luego ONCE evaluaciones de un tiron, luego siete mas, luego
# ranking. Eso es lo que ejercita el `for pedido in pedidos` del bucle, que
# esta escrito desde el paso 2 y aqui por fin tiene trabajo.
#
# Y mira los tokens enviados: 1159 en la vuelta 1 y 4337 en la ultima. Si este
# agente corriera contra un Ollama con la ventana por omision -4096- la ultima
# vuelta YA NO CABRIA. Se comenta debajo.
agentes.experimentador("Evalua todos los predictores del catalogo y reporta sus metricas.",
                       detalle=True, ver=False)


### La tool que no se usaba

Esta sección estaba escrita al revés hasta que se midió, y vale la pena
contarla como pasó.

**Antes**, el Experimentador tenía solo dos tools —`listar_predictores` y
`evaluar_predictor`— y **nunca llamaba a la primera**: iba directo a evaluar. La causa no era el modelo. La
descripción del parámetro `predictor` enumeraba los once nombres válidos:

> *predictor*: `extra_trees, svr, random_forest, knn, linear, ridge, ...`

El catálogo ya estaba escrito en el esquema de la otra tool, y ese esquema
viaja en el mismo request que tu pregunta. **Nadie pide un catálogo que ya
tiene a la vista.**

El arreglo fue sacar la enumeración y dejar en su lugar un puntero —*"los
nombres válidos salen de `listar_predictores`: llámala primero"*:

| descripción de `evaluar_predictor` | llamó a `listar_predictores` |
|---|---|
| enumerando los once nombres | **0 de 4** |
| apuntando a la otra tool (como está hoy) | **4 de 4**, siempre como primera llamada |

Y la tool pasó a ganarse el lugar: ahora devuelve también `faltan_por_evaluar`,
que cambia en cada llamada, y `no_disponibles`, que depende de lo que tengas
instalado. Eso **no se puede** escribir en un esquema fijo.

**La regla que queda:** lo que pongas en la descripción de una tool decide qué
tools llega a llamar el agente. Es el hallazgo del bloque 2 visto desde el otro
lado — allá era *cuánto devuelve* una tool lo que decidía cuántas vueltas da.
Ninguno de los dos se descubre leyendo el código: se descubre midiendo.


### Y una tool que apareció midiendo

Con los 18 resultados en el contexto, se le pidió cerrar con *"los cinco
mejores por c_index"*. Los evaluó bien los 18 —los números son suyos— y armó
mal el ranking **3 veces de 3**: siempre se saltaba `xgboost`, que es el cuarto.

| | 1º | 2º | 3º | 4º | 5º |
|---|---|---|---|---|---|
| **real** | hist_gb | svr | extra_trees | **xgboost** | random_forest |
| **lo que dijo, 3/3** | hist_gb | svr | extra_trees | ~~random_forest~~ | ~~bagging~~ |

No es falta de datos. Es que **ordenar dieciocho números no es algo que un LLM
haga de forma confiable** — el mismo hallazgo del bloque 2, donde el agente no
sabía multiplicar 1847 × 263.

El arreglo no fue insistirle en el prompt. Fue darle `ranking()`, nueve líneas
de Python con un `sorted()`, y decirle en el system prompt que no ordene él.

**Y en la corrida de arriba se ve funcionando.** En la vuelta 4 pide
`ranking({"top": 30})`, y la tabla con la que cierra es la que le devolvió esa
llamada: `hist_gradient_boosting`, `svr`, `extra_trees`, `xgboost`… con
`xgboost` en el cuarto puesto, que es justo el que fallaba cuando ordenaba él.

> Es la división de trabajo de todo el taller: lo que la máquina hace exacto,
> que lo haga la máquina; el modelo se queda con lo que solo él puede hacer, que
> es decidir qué pedir y cómo contarlo.

> **Que la llame no está garantizado.** En otra corrida, con el mismo pedido,
> cerró **sin** llamarla y ordenó él los dieciocho: acertó los tres primeros y
> metió tres inversiones en la cola —`knn` con `mlp`, `lasso` con `elasticnet`,
> y `bayesian_ridge` debajo de `linear` y `ridge`—. Tener la tool no basta: lo
> que decide que se llame es el pedido más el system prompt.

> Y de paso, en la traza se ve algo que no se adivina leyendo el código: el
> modelo pide **once evaluaciones en una sola vuelta**, y siete en la
> siguiente. Las 20 llamadas a tools caben en **cinco vueltas**.

### La tool más cara: `buscar_hiperparametros`

Todas las métricas de arriba salen de los **valores por omisión** de
scikit-learn. El paper no hizo eso: su Tabla 2 publica la rejilla de cada uno de
los once modelos. Así que el Experimentador tiene una cuarta tool que prueba la
rejilla y devuelve solo la mejor configuración.

Es la tool más cara del sistema —entre 1 y 60 segundos según el predictor— y su
descripción se lo dice: *"úsala solo en los que ya quedaron arriba del ranking,
no en todos"*. Las once rejillas enteras son 129 segundos de CPU.

Qué gana el ajuste, medido:

| predictor | por omisión | mejor de la rejilla | el paper |
|---|---|---|---|
| `knn` | 0.5686 | **0.6595** | 0.6542 |
| `xgboost` | 0.6620 | **0.6778** | 0.6737 |
| `random_forest` | 0.6582 | **0.6739** | 0.6726 |
| `svr` | 0.6732 | **0.6766** | 0.6739 |
| `extra_trees` | 0.6679 | 0.6767 | 0.6771 |
| `adaboost` | 0.5568 | 0.5570 | 0.5618 |

`knn` es el caso que vale contar: con los valores por omisión queda 0.086 abajo
del paper y parece un predictor malo. Con `n_neighbors=7, weights='distance',
p=1` sube a 0.6595 y **le gana al paper**. Los valores por omisión de
scikit-learn castigan a KNN mucho más que a los árboles, y eso no es una
propiedad del método: es una propiedad de una configuración que nadie revisó.

**Y lo que importa para lo que sigue:** una configuración que gana se anota en
el pizarrón como una **fila nueva**, con el nombre `predictor/entero+ajustado`.
Es la razón de que la primera corrida del Orquestador acabe comparando **20
evaluaciones y no 18**, y de que el grupo de empatados salga distinto.

## 3. El Estadístico

Su trabajo es desconfiar. Corre la prueba de Friedman sobre todo lo que el
Experimentador midió y dice si las diferencias caben en el azar.

In [ ]:
# La ficha del Estadistico. Lo que hay que notar es lo que NO aparece en su
# lista: no tiene describir_dataset ni evaluar_predictor. No puede mirar los
# datos ni entrenar nada.
#
# Solo ve lo que el Experimentador ya dejo en el pizarron. Esa limitacion es
# el agente: un revisor que pudiera volver a correr los experimentos acabaria
# defendiendo sus propios numeros.
agentes.ficha("estadistico")


In [ ]:
# Pocas vueltas: pide la prueba de Friedman y redacta. Con detalle=True se ve
# que lo unico que recibe es el pizarron -nunca los datos-, que es justo lo
# que lo vuelve un control y no un segundo experimentador.
agentes.estadistico(detalle=True, ver=False)


### Lo que dijo, contra lo que le devolvió la tool

Vale pararse aquí, porque es el agente del que más se depende y acaba de cometer
el error de la casa.

La tool le devolvió **dos** grupos, porque son dos criterios distintos: la
heurística —rangos medios a menos de 1.0 del mejor— nombra a cuatro
(`hist_gradient_boosting`, `svr`, `extra_trees`, `xgboost`), y el post-hoc de
Nemenyi, que es el del diagrama del paper, nombra a **diez de los dieciocho**.

Y él dijo: *"los 18 predictores son indistinguibles entre sí según el criterio
conservador de Nemenyi"*. **Dieciocho, no diez.**

La dirección del error es la prudente, y la conclusión práctica que saca
—desempatar por tiempo de entrenamiento— es la correcta. Pero dieciocho no
estaba en ninguna de las dos listas: lo redondeó a "todos".

Es la tercera forma del mismo hallazgo. **Lo que el agente hace bien es decidir
qué pedir y cómo contarlo; lo que hace mal es manipular las cifras que le
devuelven.** No calcula —eso es el bloque 2—, no ordena —la sección del
Experimentador— y no copia una lista de diez nombres sin redondearla.

### Por qué este agente existe

El paper mismo lo necesitó. Su Figura 2 —el diagrama de diferencia crítica—
muestra que **Extra Trees, SVR, XGBoost, Random Forest y KNN están en el mismo
grupo estadístico**: no son distinguibles.

O sea que "Extra Trees es el mejor" es, estrictamente, una afirmación de más.

Es el guardrail del paso 4 aplicado a **resultados** en vez de a **acciones**.
Allá impedíamos que el agente borrara algo; aquí le impedimos que afirme de más.

<img src="imagenes/16-estadistico.png" width="560">

**No mira los datos: mira los resultados de los otros.** Revisa con pluma roja
la misma tira que salió de la máquina del Experimentador —que está al fondo,
borrosa, y que él no toca— y ya tachó varias líneas.

Fíjate en el trofeo de primer lugar: lo rodeó con un círculo rojo y le puso un
signo de interrogación. Ese signo de interrogación es la Figura 2 del paper.


## 4. El Orquestador

Ahora los tres juntos. Tres de sus cuatro tools son los otros agentes, así que
vas a ver el handoff: `>>> el orquestador delega en ...`. La cuarta es
`ranking`, y está ahí porque pedirle el orden completo al Experimentador no
funciona — es el hallazgo de la sección anterior.

**Tarda entre 4 y 6 minutos.** Son cuatro agentes, y el Experimentador solo ya
se lleva dos y medio barriendo los 18 predictores — más, si además decide
ajustarle los hiperparámetros a alguno, que es lo que pasó en la corrida
guardada aquí.

<img src="imagenes/17-orquestador.png" width="560">

**Las manos vacías.** No puede medir nada: ni mirar los datos, ni entrenar, ni
correr una prueba estadística. Los tubos que le salen del cinturón hacia los
otros agentes son casi todo lo que tiene — lo único que puede hacer él es
`ranking`, que ordena lo que los otros ya midieron. La hoja que sostiene está
en blanco hasta que ellos la llenen.

Y los tubos están enredados. Coordinar a tres especialistas es el trabajo, no un
detalle de implementación.


In [ ]:
# La ficha del Orquestador es la mas rara de las cuatro: tres de sus cuatro
# tools no son funciones de laboratorio, son los otros tres agentes. Eso es el
# patron agente-como-tool, y aqui se ve sin tener que explicarlo.
#
# La cuarta, ranking, es la excepcion y vale leer su descripcion: lee el
# pizarron, pero solo ORDENA lo que los otros ya midieron. Medir -mirar los
# datos, entrenar, correr la prueba- no puede.
agentes.ficha("orquestador")


In [ ]:
# detalle=True aqui muestra las vueltas DEL ORQUESTADOR: vas a ver que lo
# unico que recibe de vuelta es el texto de cada especialista, nunca un numero
# medido por el mismo.
#
# Los tres subagentes siguen en modo compacto -ya los viste en detalle arriba,
# y los cuatro a la vez serian ilegibles-, pero ahora tambien marcan sus
# vueltas.
#
# Esta peticion le pide dos de las tres cosas que sabe contestar: CUAL es el
# mejor, y si el resultado es SOLIDO. Las dos tienen que aparecer en su
# respuesta final.
lab.limpiar_pizarron()
agentes.orquestador(
    "Encuentra el mejor predictor de PSNR para este dataset de arquitecturas, "
    "y dime si el resultado es solido.",
    detalle=True
)


In [ ]:
# La misma maquinaria con otra peticion. Tres cosas que mirar, y la tercera
# es una falla.
#
# 1. El pizarron arranca limpio, asi que en la vuelta 1 llama a ranking y le
#    vuelve un error -"el pizarron esta vacio; evalua algo primero"-. Se
#    corrige solo: en la vuelta 2 delega en el Experimentador. Es el bucle
#    devolviendole el error al modelo en vez de tumbarse, como en el paso 3.
#
# 2. Aqui el Experimentador SI llama a ranking, porque el pedido pide el orden
#    completo. Comparalo con lo que armo a mano dos secciones arriba: los 18
#    salen bien ordenados, de arriba a abajo.
#
# 3. Su system prompt dice "no concluyas sin haber consultado al estadistico".
#    Fijate si lo consulta. En la corrida guardada aqui SI lo hizo, pero en
#    una corrida anterior no, y la conclusion cambio por eso. Se discute debajo.
lab.limpiar_pizarron()
agentes.orquestador(
    "Dame el ranking de los predictores, y dime cual es la metrica mas importante para decidir cual es el mejor.",
    detalle=True
)


### Lo que pasó en las dos corridas

Son dos peticiones distintas a la misma maquinaria. **Las dos delegaron en los
tres especialistas en orden** —analista, experimentador, estadístico— y las dos
se negaron a nombrar un ganador único. Pero no hicieron lo mismo, y la
diferencia no está en el código.

#### La primera: con hiperparámetros ajustados

El Experimentador evaluó los 18, llamó a `ranking`, y además **decidió por su
cuenta ajustarle los hiperparámetros a los tres de arriba**. Las dos
configuraciones que ganaron entraron al pizarrón como filas nuevas, así que el
Estadístico acabó comparando **20 evaluaciones, no 18**.

Con esas 20, la heurística de rangos medios deja un grupo de dos:
`extra_trees/entero+ajustado` (3.5) y `svr/entero+ajustado` (4.2). Y el
Orquestador cerró nombrando exactamente esos dos:

> *"No hay un único ganador claro; extra_trees y svr (ambos ajustados) comparten
> el primer puesto de manera equivalente. Para desempatar, se recomienda
> considerar criterios secundarios como el tiempo de entrenamiento."*

**Copió el grupo tal como se lo dio la tool**, sin agregar ni quitar nombres.

#### La segunda: sin ajustar, y el grupo de cuatro

Aquí el Experimentador no ajustó nada: 18 evaluaciones limpias. Con esas, la
heurística nombra a cuatro, y el Orquestador los reportó:

> *"No existe un ganador claro: hist_gradient_boosting y svr son
> estadísticamente indistinguibles (diferencia de solo 0.0025 en c_index),
> formando un grupo superior junto con extra_trees y xgboost."*

El 0.0025 es exacto: 0.6757 contra 0.6732. **Eso es lo que un pipeline ingenuo
—`max(resultados, key=c_index)`— no haría:** ese habría dicho "gana
hist_gradient_boosting" y se habría quedado tan tranquilo.

#### Lo que esto enseña, y es más fino que "funciona"

**El mismo sistema, dos veces, dio dos respuestas distintas y las dos
correctas.** La diferencia no fue el azar del modelo: fue que en una corrida el
Experimentador decidió ajustar hiperparámetros y en la otra no. Esa decisión
cambia qué filas hay en el pizarrón, y eso cambia el resultado de la prueba
estadística.

Es una propiedad incómoda de los sistemas con agentes: **el resultado depende de
decisiones intermedias que nadie pidió explícitamente.** Un pipeline fijo no
tiene ese problema, y tampoco tiene la flexibilidad.

> **Si en tu corrida salió distinto**, las preguntas útiles son: ¿delegó en los
> tres? ¿ajustó hiperparámetros, o sea cuántas filas comparó? ¿nombró un
> ganador único, y si lo hizo, consultó al Estadístico y lo ignoró o ni lo
> consultó? En una corrida anterior de este mismo notebook, la segunda petición
> **se saltó al Estadístico** y nombró un ganador único por 0.0001 de
> diferencia, pese a que su system prompt dice literalmente *"no concluyas sin
> haber consultado al estadistico"*. Es una instrucción en texto, no un
> contrato: nada en el bucle la hace cumplir. Si quieres que sea obligatoria, se
> programa con un `if`, y esa es la diferencia entre un guardrail y un ruego.

### Cuál es el empate, entonces

Depende de con cuáles filas se corra la prueba, y las dos corridas lo muestran.
Con **las 18 originales** —la celda del Estadístico suelto y la segunda corrida—
el grupo son cuatro:

| | rango medio |
|---|---|
| hist_gradient_boosting | 3.4 |
| svr | 3.8 |
| extra_trees | 4.0 |
| xgboost | 4.2 |
| ~~gaussian_process~~ | 4.9 — ya fuera del grupo |

Y uno de los cuatro, `hist_gradient_boosting`, **no está en el paper**: es de
los siete que agrega este laboratorio. El empate no reproduce la Figura 2, la
cruza.

Con las **20 de la primera corrida** el mismo criterio deja solo dos: ajustar
dos modelos los separa del resto y rompe el empate de cuatro.

Nada de eso es inestabilidad del modelo: **son filas distintas en la tabla de
Friedman**. Quien decidió que fueran distintas fue el agente, al elegir ajustar
tres predictores que nadie le pidió ajustar.

### Y el empate es más grande que cualquiera de esos números

Los grupos de arriba salen de una **heurística**: rangos medios a menos de 1.0
del mejor. No es un test.

El post-hoc que corresponde a Friedman —y el que dibuja el diagrama de
diferencia crítica del paper— es **Nemenyi**, y con 18 modelos y 10 folds su
distancia crítica vale **8.33 rangos**, sobre un rango total de 12. Por ese
criterio son **diez de los dieciocho** los indistinguibles del mejor: todo hasta
`mlp`. Con las 20 filas de la primera corrida la distancia sube a 9.38 y el
grupo a **once de veinte**.

| criterio | con las 18 | con las 20 |
|---|---|---|
| umbral de 1.0 (heurística) | 4 | 2 |
| **Nemenyi, α = 0.05** | **10** | **11** |

La tool reporta los dos, en `indistinguibles_del_mejor` y en `nemenyi`.

> Dos advertencias para quien quiera usar esto en un paper. Nemenyi tiene poca
> potencia con pocas repeticiones: separar dos modelos cuesta mucho, y por eso
> el grupo sale grande. Y supone muestras independientes, que los folds de una
> validación cruzada **no** son, porque comparten datos de entrenamiento. Usar
> folds como repeticiones es lo que se hace en la práctica —y lo que permite
> comparar con el paper—, pero infla la confianza.

## Calificar al agente

Esto es lo que casi nunca se puede hacer con un agente: **comparar su respuesta
contra una verdad publicada**.

Aplica solo a los **once de la Tabla 3**. Los siete agregados no tienen contra
qué compararse, y uno de ellos es el que gana.

Para leer la columna de diferencias hay que saber una cosa: **las filas por
omisión no son comparables de igual a igual con el paper**, porque el paper
ajustó cada modelo con la rejilla de su Tabla 2. Casi todas caen dentro de
±0.01 de todos modos, pero `knn` sale 0.086 abajo — y eso no dice que KNN sea
peor de lo que el paper reporta, dice que los valores por omisión de
scikit-learn lo castigan.

Las filas `+ajustado`, en cambio, sí son comparables, y ahí se ve lo que vale
la pena ver: `extra_trees` ajustado queda a **0.0004** del paper. Reprodujimos
su número.

In [ ]:
# Ordenado por c_index, y con los siete que el paper no probo marcados: la
# comparacion solo aplica a los once de la Tabla 3.
#
# Lee el pizarron, asi que necesita que el Experimentador haya corrido antes.
# Si sale vacia, corre su celda o la del Orquestador.
#
# Un predictor puede salir DOS veces: la fila por omision y la '+ajustado' que
# anoto buscar_hiperparametros. Contra el paper -que si ajusto- la que vale es
# la ajustada, y se ve que la diferencia se cierra casi del todo.
print(f"  {'predictor':<26} {'agente':>8} {'paper':>8}   diferencia")
print(f"  {'-'*26} {'-'*8} {'-'*8}   {'-'*20}")
for e in sorted(lab.PIZARRON, key=lambda e: -e["c_index"]):
    if e["codificacion"] != "entero":
        continue
    nombre = e["predictor"] + (" +ajustado" if e["hiperparametros"] else "")
    p = lab.PAPER.get(e["predictor"])
    if p:
        d = e["c_index"] - p["c_index"]
        print(f"  {nombre:<26} {e['c_index']:>8.4f} {p['c_index']:>8.4f}   {d:+.4f}")
    else:
        print(f"  {nombre:<26} {e['c_index']:>8.4f} {'--':>8}   el paper no lo probo")


## Lo que el paper dejó abierto, y el agente puede barrer

El paper usa los 28 tokens **como enteros**, y dice explícitamente:

> *"Exploring alternative encodings (e.g., one-hot or graph-based) would require
> modifying the underlying mutation and crossover operators, which is beyond the
> scope of this work but represents a promising avenue for future work."*

Eso es terreno libre y cuesta segundos. El laboratorio tiene tres codificaciones:
`entero` (la del paper), `onehot` (28×8 = 224 columnas) y `binario` (Gray → 84
bits).

In [ ]:
for cod in lab.CODIFICACIONES:
    r = lab.evaluar_predictor("extra_trees", codificacion=cod)
    print(f"  {cod:<9} C-index {r['c_index']:.4f}   MSE {r['mse']:>6.2f}   {r['segundos']}s")

**Gana la del paper.**

La hipótesis razonable era que los códigos Gray perjudicarían: el valor 5 no
está "entre" el 4 y el 6 en ningún sentido semántico. Resultó falsa, y por dos
motivos que se entienden una vez medidos:

- Los **árboles parten por umbrales** y pueden aislar cualquier valor con
  suficientes cortes. El orden numérico les importa poco.
- El **one-hot empeora** porque son 224 columnas dispersas para 1193 filas.

Que la exploración *confirme* en vez de derribar es un resultado honesto, y
probablemente el más común en investigación.

## Y ahora la pregunta del bloque 5

> Comparado con un solo agente que tuviera las ocho tools, **¿esto mejoró algo,
> o solo se volvió más caro?**

Las cuentas de la primera corrida, para empezar la discusión: **14 vueltas al
modelo** — 5 del Orquestador, 2 del Analista, 5 del Experimentador y 2 del
Estadístico. Cuatro contextos separados, cada uno pagando su system prompt
completo en cada vuelta.

**Interpretación, discutible:**

**Lo que el reparto sí compró.** El Estadístico tiene un system prompt que le
dice que desconfíe, y **no puede entrenar nada**: solo ve el pizarrón. Un
agente único con las ocho tools tiende a quedarse con su propio resultado: el
que mide es el que juzga. Separarlos no es cosmético — es la misma idea que
separar quien escribe de quien aprueba.

**Y lo que no compró, que lo viste arriba.** La segunda corrida se saltó al
Estadístico y nombró un ganador único. Separar los roles pone el control
*disponible*; no lo vuelve *obligatorio*. Eso último cuesta un `if`, no un
agente más.

**Lo que costó.** Cuatro agentes son cuatro contextos, cuatro system prompts y
varias llamadas más al modelo. Para este problema, un agente único con un buen
prompt probablemente llegaría a un resultado parecido, más rápido.

**Dónde sí hace falta.** Cuando el juicio y la ejecución tienen que estar
separados por diseño, no por estilo. Aquí el Estadístico es un control, igual
que la confirmación humana del paso 4.

## Tu turno

- Quítale al Analista la línea del `NUNCA respondas sin...` y vuelve a correrlo.
  ¿Inventa?
- Cámbiale al Experimentador la métrica: dile que lo que importa es el `mse`.
  ¿Elige otro modelo?
- Añade un predictor al laboratorio y deja que el agente lo descubra con
  `listar_predictores`.
- Haz lo mismo con **un solo agente** que tenga las ocho tools, y compara
  vueltas, tiempo y calidad de la conclusión.
- Haz obligatorio al Estadístico: un `if` en el bucle del Orquestador que no
  deje cerrar sin haber llamado `consultar_estadistico`. Es la falla de la
  segunda corrida, y se arregla con cuatro líneas de Python.